In [2]:
# Imports and Environment Setup 

import os
import time
from dotenv import load_dotenv

# Import LangChain components
from langchain_community.embeddings import OllamaEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate

# Load the API key from the .env file
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

# Check if the API key is available
if not api_key:
    raise ValueError("Google API Key not found. Please set it in the .env file.")
else:
    print("Libraries imported and API key loaded successfully.")

e:\HealthCareProject\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported and API key loaded successfully.


In [3]:
#  Langsmth tracking 
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "Healthcare Chatbot" 

In [4]:
# Constants and Custom Prompt Template

# Define paths for documents and vector store
PDF_DOCS_PATH = "./documents"
CHROMA_PERSIST_PATH = "./db/chroma_store"

# prompt for the LLM's behavior.
custom_prompt_template = """
You are a helpful and honest medical information assistant.
Your task is to provide answers based on the provided context.
You can also provide answer based on your knowledge
Your answers should be clear and concise.
Do not mention that you are getting the information from a provided text.

Context: {context}
Chat History: {chat_history}

Question: {question}
Helpful Answer:
"""

def create_custom_prompt():
    """Creates a custom prompt template for the RAG chain."""
    return PromptTemplate(
        template=custom_prompt_template,
        input_variables=["context", "chat_history", "question"]
    )

print("Constants and prompt template function defined.")

Constants and prompt template function defined.


In [5]:
# Vector Store Function(with Ollama)

def get_vectorstore():
    """
    Processes PDF documents using a local Ollama model for embeddings.
    Creates or loads a persistent Chroma vector store.
    """
    # Using Ollama model for creating embeddings
    embeddings = OllamaEmbeddings(model="llama3.2:1b")

    if os.path.exists(CHROMA_PERSIST_PATH):
        # Load the existing vector store
        print("Loading existing vector store...")
        vector_store = Chroma(persist_directory=CHROMA_PERSIST_PATH, embedding_function=embeddings)
    else:
        # Create a new vector store
        print("Creating new vector store...")
        
        pdf_files = [f for f in os.listdir(PDF_DOCS_PATH) if f.endswith(".pdf")]
        if not pdf_files:
            raise FileNotFoundError(f"No PDF files found in '{PDF_DOCS_PATH}'.")

        documents = []
        for pdf_file in pdf_files:
            loader = PyPDFLoader(os.path.join(PDF_DOCS_PATH, pdf_file))
            documents.extend(loader.load())
        
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
        chunks = text_splitter.split_documents(documents)
        
        # Create and persist the vector store using the loaded chunks
        vector_store = Chroma.from_documents(chunks, embeddings, persist_directory=CHROMA_PERSIST_PATH)
        print("Vector store created successfully.")
    
    return vector_store

print("Vector store function with Ollama embeddings defined.")

Vector store function with Ollama embeddings defined.


In [5]:
# Conversational Chain Function(with Gemini)

def get_conversational_rag_chain(vector_store):
    """
    Creates the main conversational retrieval chain using Google Gemini for generation.
    """
    llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash-latest", google_api_key=api_key, temperature=0.3)

    # Set up memory to store chat history
    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True
    )

    # Create the retriever from the vector store
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})

    # Create the custom prompt
    prompt = create_custom_prompt()

    # Create the conversational chain
    chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=retriever,
        memory=memory,
        combine_docs_chain_kwargs={"prompt": prompt},
        verbose=False
    )
    
    return chain

print("Conversational RAG chain function with Gemini model defined.")

Conversational RAG chain function with Gemini model defined.


In [6]:
vector_store = get_vectorstore()


C:\Users\akmal\AppData\Local\Temp\ipykernel_34752\3825764663.py:9: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="llama3.2:1b")
C:\Users\akmal\AppData\Local\Temp\ipykernel_34752\3825764663.py:14: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vector_store = Chroma(persist_directory=CHROMA_PERSIST_PATH, embedding_function=embeddings)


Loading existing vector store...


In [7]:
qa_chain = get_conversational_rag_chain(vector_store)

C:\Users\akmal\AppData\Local\Temp\ipykernel_34752\1168976536.py:10: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


In [1]:
# Interactive Chat

# Initialize chat history once.
try:
    chat_history
except NameError:
    chat_history = []

# --- Ask qn ---
question = "what is the symptoms of fever "
# --------------------------

if question.lower() == 'exit':
    print("Chat ended.")
else:
    # Get the result from the chain
    result = qa_chain({"question": question, "chat_history": chat_history})
    
    # Append the current question and the answer to the history
    chat_history.append((question, result["answer"]))
    
    # Print the answer
    print("AI:", result["answer"])

NameError: name 'qa_chain' is not defined

In [9]:
import json

In [10]:

HISTORY_FILE = "chat_history.json"

def save_chat_history(history, file_path=HISTORY_FILE):
    """Saves the chat history to a JSON file."""
    with open(file_path, 'w') as f:
        json.dump(history, f, indent=4)

def load_chat_history(file_path=HISTORY_FILE):
    """Loads the chat history from a JSON file."""
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            return json.load(f)
    return []

In [11]:
# Interactive Chat with history

# Load the history from the file 
chat_history = load_chat_history()
print(f"Loaded {len(chat_history)} items from previous conversations.")

# Ask qn
question = "What was the last disease I asked you about?"

if question.lower() == 'exit':
    print("Chat ended.")
else:
    # Get the result from the chain
    result = qa_chain({"question": question, "chat_history": chat_history})
    
    # Append the current question and the answer to the history
    chat_history.append((question, result["answer"]))
    
    # --- Save the updated history back to the file ---
    save_chat_history(chat_history)
    
    # Print the answer
    print("\nAI:", result["answer"])

Loaded 0 items from previous conversations.


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-1.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 50
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 17
}
].
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 4.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing de

KeyboardInterrupt: 

In [6]:
import os
import google.generativeai as genai
from dotenv import load_dotenv
from google.api_core import exceptions

print("--- Starting Google Gemini API Key Test ---")

# 1. Load the environment variables from your .env file
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

# 2. Check if the API key was found
if not api_key:
    print("\n[ERROR] GOOGLE_API_KEY not found in .env file.")
    print("Please make sure your .env file is in the same directory and contains the key.")
else:
    print("\nAPI Key found in .env file.")
    try:
        # 3. Configure the generative AI client with your key
        genai.configure(api_key=api_key)

        # 4. Create the model
        # We use a known stable model name for this test.
        model = genai.GenerativeModel('gemini-2.5-flash')
        print("Successfully created a Gemini 1.0 Pro model instance.")

        # 5. Send a test prompt
        print("Sending a test prompt to the API...")
        prompt = "Why is the sky blue? Answer in one sentence."
        response = model.generate_content(prompt)

        # 6. Print the response
        print("\n--- TEST SUCCESSFUL ---")
        print(f"Prompt: {prompt}")
        print(f"Response: {response.text}")

    except exceptions.PermissionDenied as e:
        print("\n--- TEST FAILED: Permission Denied (403) ---")
        print("This error means your API key is likely correct, but it doesn't have the necessary permissions.")
        print("ACTION: Go to your Google Cloud project and make sure the 'Generative Language API' or 'Vertex AI API' is ENABLED.")
        print(f"Full error details: {e}")
        
    except exceptions.NotFound as e:
        print("\n--- TEST FAILED: Not Found (404) ---")
        print("This error means the model name might be incorrect for your region or account type.")
        print(f"Full error details: {e}")

    except Exception as e:
        print(f"\n--- TEST FAILED: An unexpected error occurred ---")
        print("This could be due to an invalid API key, network issues, or other problems.")
        print(f"Full error details: {e}")

--- Starting Google Gemini API Key Test ---

API Key found in .env file.
Successfully created a Gemini 1.0 Pro model instance.
Sending a test prompt to the API...

--- TEST SUCCESSFUL ---
Prompt: Why is the sky blue? Answer in one sentence.
Response: The sky is blue because Earth's atmosphere scatters blue light from the sun more effectively than other colors, making the blue light visible from all directions.


In [5]:
import google.generativeai as genai
import os
from dotenv import load_dotenv

load_dotenv()
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

for model in genai.list_models():
    print(model.name, model.supported_generation_methods)


models/embedding-gecko-001 ['embedText', 'countTextTokens']
models/gemini-2.5-pro-preview-03-25 ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-flash-preview-05-20 ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-flash ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-flash-lite-preview-06-17 ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-pro-preview-05-06 ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-pro-preview-06-05 ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-pro ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-exp ['generateContent', 'countTokens', 'bidiGenerateContent']
models/gemini-2.0-flash ['generateContent', '